# FGSM Adversarial Attack & Defense — Experiment Notebook

> **Objective:** Demonstrate how a trained CNN can be fooled by imperceptible perturbations (FGSM attack) and how Adversarial Training restores robustness.

| Section | Description |
|---------|-------------|
| 1 | Setup & Data Loading |
| 2 | CNN Training |
| 3 | FGSM Attack |
| 4 | Visualising Adversarial Examples |
| 5 | Adversarial Training Defense |
| 6 | Comparison & Conclusions |

## Background — What is FGSM?

FGSM (Goodfellow et al., 2014) creates adversarial examples in a single step:

$$x_{adv} = x + \varepsilon \cdot \text{sign}(\nabla_x J(\theta, x, y))$$

- $x$ — clean input image  
- $y$ — true label  
- $J$ — cross-entropy loss  
- $\varepsilon$ — perturbation magnitude (attack strength)  
- $\nabla_x J$ — gradient of loss with respect to input pixels

The attack moves each pixel by a tiny step `ε` in the direction that **maximises** the model's loss — making it misclassify the image while keeping the change invisible to humans.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.metrics import confusion_matrix, classification_report

from attacks.fgsm import fgsm_attack, evaluate_attack
from defenses.adversarial_training import adversarial_train

np.random.seed(42)
tf.random.set_seed(42)
print('TensorFlow:', tf.__version__)

## 1 — Data Loading

In [ ]:
data = np.load('../dataset/data.npz')
x_train, y_train_raw = data['x_train'], data['y_train']
x_test,  y_test_raw  = data['x_test'],  data['y_test']

y_train = tf.keras.utils.to_categorical(y_train_raw, 10)
y_test  = tf.keras.utils.to_categorical(y_test_raw,  10)

print(f'Train: {x_train.shape}  |  Test: {x_test.shape}')
print(f'Pixel range: [{x_train.min():.2f}, {x_train.max():.2f}]')

# Show one sample per digit class
fig, axes = plt.subplots(1, 10, figsize=(18, 2))
for d in range(10):
    idx = np.where(y_train_raw == d)[0][0]
    axes[d].imshow(x_train[idx, :, :, 0], cmap='gray')
    axes[d].set_title(str(d), fontsize=12)
    axes[d].axis('off')
plt.suptitle('One Sample per Digit Class', y=1.05, fontsize=12)
plt.tight_layout()
plt.show()

## 2 — Train the CNN

In [ ]:
def build_cnn():
    model = tf.keras.Sequential([
        tf.keras.layers.Input(shape=(28, 28, 1)),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(32, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.Conv2D(64, 3, activation='relu', padding='same'),
        tf.keras.layers.MaxPooling2D(),
        tf.keras.layers.Dropout(0.25),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(256, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation='softmax'),
    ])
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

MODEL_PATH = '../models/cnn_model.h5'
if os.path.exists(MODEL_PATH):
    print('Loading saved model …')
    model = tf.keras.models.load_model(MODEL_PATH)
else:
    model = build_cnn()
    model.summary()
    cb = [tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)]
    history = model.fit(x_train, y_train, epochs=30, batch_size=64,
                        validation_split=0.15, callbacks=cb)
    model.save(MODEL_PATH)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(history.history['accuracy'],     label='Train')
    axes[0].plot(history.history['val_accuracy'], label='Val')
    axes[0].set_title('Accuracy'); axes[0].legend(); axes[0].grid(alpha=.3)
    axes[1].plot(history.history['loss'],         label='Train')
    axes[1].plot(history.history['val_loss'],     label='Val')
    axes[1].set_title('Loss'); axes[1].legend(); axes[1].grid(alpha=.3)
    plt.suptitle('Training Curve', fontweight='bold'); plt.tight_layout(); plt.show()

_, acc = model.evaluate(x_test, y_test, verbose=0)
print(f'\nClean test accuracy: {acc*100:.2f}%')

## 3 — Apply FGSM Attack

In [ ]:
epsilons = [0.0, 0.05, 0.10, 0.15, 0.20, 0.25, 0.30, 0.35, 0.40]
print('Evaluating FGSM at different epsilon values …')
clean_accs = evaluate_attack(model, x_test, y_test, epsilons)

plt.figure(figsize=(8, 5))
plt.plot([e*100 for e in epsilons], [a*100 for a in clean_accs],
         'o-', color='#E74C3C', lw=2.5, ms=8, label='Standard CNN')
plt.fill_between([e*100 for e in epsilons], [a*100 for a in clean_accs],
                 10, alpha=0.08, color='#E74C3C')
plt.axhline(10, color='gray', linestyle='--', alpha=0.5, label='Random guess')
plt.xlabel('Perturbation Strength ε (×100)', fontsize=12)
plt.ylabel('Test Accuracy (%)', fontsize=12)
plt.title('FGSM Attack: Accuracy vs Epsilon', fontsize=13, fontweight='bold')
plt.legend(); plt.grid(alpha=.3); plt.show()

## 4 — Visualising Adversarial Examples

In [ ]:
sample_idx = [np.where(y_test_raw == d)[0][0] for d in range(10)]
x_sample = x_test[sample_idx]
y_sample = y_test[sample_idx]

eps_val = 0.25
adv = fgsm_attack(model, x_sample, y_sample, eps_val)
orig_preds = np.argmax(model.predict(x_sample, verbose=0), axis=1)
adv_preds  = np.argmax(model.predict(adv,       verbose=0), axis=1)

fig, axes = plt.subplots(2, 10, figsize=(20, 5))
for i in range(10):
    axes[0, i].imshow(x_sample[i,:,:,0], cmap='gray'); axes[0, i].axis('off')
    axes[0, i].set_title(f'True:{y_test_raw[sample_idx[i]]}\nPred:{orig_preds[i]}', fontsize=7)
    axes[1, i].imshow(adv[i,:,:,0], cmap='gray'); axes[1, i].axis('off')
    c = '#E74C3C' if adv_preds[i] != y_test_raw[sample_idx[i]] else '#27AE60'
    axes[1, i].set_title(f'Adv:{adv_preds[i]}', fontsize=7, color=c)
fig.text(0.01, 0.75, 'Original',    va='center', fontsize=10, fontweight='bold')
fig.text(0.01, 0.25, 'Adversarial', va='center', fontsize=10, fontweight='bold', color='#E74C3C')
plt.suptitle(f'FGSM Samples (ε={eps_val}) — Red=Fooled, Green=Correct', fontsize=11)
plt.tight_layout(); plt.show()

# Perturbation heatmap
fig, axes = plt.subplots(3, 10, figsize=(20, 7))
adv_mid = fgsm_attack(model, x_sample, y_sample, 0.15)
for i in range(10):
    axes[0,i].imshow(x_sample[i,:,:,0], cmap='gray'); axes[0,i].axis('off')
    axes[1,i].imshow(np.abs(adv_mid[i]-x_sample[i])[:,:,0]*10, cmap='hot'); axes[1,i].axis('off')
    axes[2,i].imshow(adv_mid[i,:,:,0], cmap='gray'); axes[2,i].axis('off')
for r,lbl in enumerate(['Original','Noise ×10','Adversarial']):
    axes[r,0].set_ylabel(lbl, fontsize=9, fontweight='bold')
plt.suptitle('Perturbation Heatmap (ε=0.15, magnified ×10)', fontsize=12)
plt.tight_layout(); plt.show()

In [ ]:
# Confusion matrix at eps=0.20
adv20 = fgsm_attack(model, x_test, y_test, 0.20)
preds = np.argmax(model.predict(adv20, verbose=0), axis=1)
cm = confusion_matrix(y_test_raw, preds)

fig, ax = plt.subplots(figsize=(8, 7))
im = ax.imshow(cm, cmap='Blues'); plt.colorbar(im)
ax.set_xticks(range(10)); ax.set_yticks(range(10))
ax.set_xlabel('Predicted'); ax.set_ylabel('True')
ax.set_title('Confusion Matrix — FGSM Attack (ε=0.20)', fontweight='bold')
thresh = cm.max()/2
for r in range(10):
    for c in range(10):
        ax.text(c, r, cm[r,c], ha='center', va='center', fontsize=8,
                color='white' if cm[r,c]>thresh else 'black')
plt.tight_layout(); plt.show()
print(classification_report(y_test_raw, preds))

## 5 — Adversarial Training Defense

In [ ]:
DEFENDED_PATH = '../models/defended_cnn.h5'
if os.path.exists(DEFENDED_PATH):
    print('Loading saved defended model …')
    defended_model = tf.keras.models.load_model(DEFENDED_PATH)
else:
    defended_model = build_cnn()
    defended_model.fit(x_train, y_train, epochs=5, batch_size=64,
                       validation_split=0.15, verbose=0)
    adv_hist = adversarial_train(defended_model, x_train, y_train,
                                  x_test, y_test, epsilon=0.15, epochs=8, batch_size=64)
    defended_model.save(DEFENDED_PATH)

    plt.figure(figsize=(8,5))
    plt.plot(adv_hist['accuracy'],     'o-', color='#3498DB', lw=2, label='Train')
    plt.plot(adv_hist['val_accuracy'], 's-', color='#27AE60', lw=2, label='Val')
    plt.title('Adversarial Training Progress', fontweight='bold')
    plt.legend(); plt.grid(alpha=.3); plt.show()

_, def_acc = defended_model.evaluate(x_test, y_test, verbose=0)
print(f'Defended model clean accuracy: {def_acc*100:.2f}%')

## 6 — Comparison & Conclusions

In [ ]:
defended_accs = evaluate_attack(defended_model, x_test, y_test, epsilons)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot([e*100 for e in epsilons],[a*100 for a in clean_accs],
             'o-', color='#E74C3C', lw=2.5, ms=8, label='Standard CNN')
axes[0].plot([e*100 for e in epsilons],[a*100 for a in defended_accs],
             's-', color='#27AE60', lw=2.5, ms=8, label='Defended CNN')
axes[0].fill_between([e*100 for e in epsilons],
                     [a*100 for a in clean_accs],[a*100 for a in defended_accs],
                     alpha=0.12, color='#27AE60', label='Robustness gain')
axes[0].axhline(10, color='gray', linestyle='--', alpha=0.4)
axes[0].set_xlabel('ε (×100)'); axes[0].set_ylabel('Accuracy (%)')
axes[0].set_title('Accuracy vs Epsilon', fontweight='bold')
axes[0].legend(); axes[0].grid(alpha=.3)

sel=[0,0.15,0.30]; x_pos=np.arange(3); w=0.35
sel_idx=[epsilons.index(e) for e in sel]
axes[1].bar(x_pos-w/2,[clean_accs[i]*100 for i in sel_idx],   w, color='#E74C3C',label='Standard CNN',alpha=0.85)
axes[1].bar(x_pos+w/2,[defended_accs[i]*100 for i in sel_idx],w, color='#27AE60',label='Defended CNN', alpha=0.85)
axes[1].set_xticks(x_pos); axes[1].set_xticklabels([f'ε={e}' for e in sel])
axes[1].set_ylabel('Accuracy (%)'); axes[1].set_title('Key Epsilon Comparison', fontweight='bold')
axes[1].legend(); axes[1].grid(axis='y', alpha=.3)

plt.suptitle('Standard vs Adversarially Trained CNN', fontsize=14, fontweight='bold')
plt.tight_layout(); plt.show()

In [ ]:
print('='*55)
print(f'{"epsilon":>10}  {"Standard":>12}  {"Defended":>12}  {"Gain":>8}')
print('-'*55)
for i, e in enumerate(epsilons):
    gain = (defended_accs[i]-clean_accs[i])*100
    print(f'  {e:8.2f}  {clean_accs[i]*100:11.2f}%  {defended_accs[i]*100:11.2f}%  {gain:+7.2f}%')
print('='*55)

## Conclusions

| Finding | Observation |
|---------|-------------|
| **Vulnerability** | Even tiny ε ≈ 0.10 causes dramatic accuracy drop in the standard CNN |
| **Invisibility** | The perturbations are imperceptible to the human eye |
| **Defense** | Adversarial training substantially restores robustness across all ε values |
| **Trade-off** | Defended model may sacrifice a small amount of clean accuracy for robustness |

### Key Takeaways
1. **FGSM** exploits the linearity of deep networks — a single gradient step is enough to fool them.
2. **Adversarial training** is the most effective and widely-used defense, but is computationally expensive.
3. No defense is perfect — stronger attacks (PGD, C&W) can still bypass adversarially trained models.
4. **Certified defenses** and **randomized smoothing** are active research areas toward provable robustness.